# Visium Cortical Structure Analysis

**Biological Question:** Can unsupervised gene expression clustering recover anatomically distinct tissue regions in mouse brain Visium data without cell type deconvolution?

**Dataset:** 10x Genomics Visium mouse brain section, loaded via Squidpy (2,688 spots, 18,078 genes)

**Tools:** Scanpy, Squidpy

**Key limitation:** Each Visium spot captures signal from multiple cell types simultaneously. Clusters reflect dominant mixture signals, not pure cell populations. Cell type identity requires deconvolution with a single cell reference such as Cell2location.

In [ ]:
# Install required packages
# Run this cell first if using Google Colab
!pip install -q scanpy squidpy leidenalg igraph

import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import os

# Configure figure output directory
os.makedirs('figures', exist_ok=True)
sc.settings.figdir = 'figures/'
sc.settings.verbosity = 1

print('Imports complete')

## 1. Load Dataset

The Squidpy built-in mouse brain Visium dataset is a publicly available 10x Genomics section. It loads directly without manual download, making it suitable for reproducible analysis on limited hardware.

In [ ]:
adata = sq.datasets.visium_hne_adata()
print(f'Loaded: {adata.n_obs} spots, {adata.n_vars} genes')

## 2. Quality Control

Low quality spots produce noise that distorts clustering. I flag mitochondrial genes as a proxy for cell damage: dying cells lose cytoplasmic RNA but retain mitochondrial RNA, so a high mitochondrial fraction indicates a compromised spot. I filter spots with fewer than 200 detected genes and remove genes detected in fewer than 10 spots.

In [ ]:
# Identify mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('mt-')

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

# Visualize QC distributions before filtering
sc.pl.violin(
    adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    save='_qc_metrics.png'
)

# Apply filters
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=10)
adata = adata[adata.obs['pct_counts_mt'] < 20]

print(f'After QC: {adata.n_obs} spots, {adata.n_vars} genes')

## 3. Normalization

Raw count differences between spots reflect capture efficiency differences, not biology. Normalizing to 10,000 counts per spot removes this technical variation. Log transformation then compresses the dynamic range so that highly expressed genes do not dominate downstream analysis.

In [ ]:
# Save raw counts before normalization for downstream use
adata.raw = adata

# Normalize to 10,000 counts per spot
sc.pp.normalize_total(adata, target_sum=1e4)

# Log transform
sc.pp.log1p(adata)

print('Normalization complete')

## 4. Feature Selection and Dimensionality Reduction

Most genes carry little information about tissue structure. Selecting the top 2,000 highly variable genes focuses the analysis on genes that differ meaningfully between spots. PCA compresses these 2,000 dimensions into a lower-dimensional representation. I compute 50 components to inspect the full variance spectrum, then use the top 30 components for the neighbor graph, capturing biological signal while excluding noise from low-variance components.

In [ ]:
# Select top 2000 highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
adata = adata[:, adata.var['highly_variable']]
print(f'Highly variable genes selected: {adata.n_vars}')

# Scale data
sc.pp.scale(adata, max_value=10)

# Run PCA with 50 components
sc.tl.pca(adata, n_comps=50)

# Visualize variance explained to confirm signal drop-off point
sc.pl.pca_variance_ratio(
    adata,
    n_pcs=50,
    save='_pca_variance.png'
)

print('PCA complete')

## 5. Neighbor Graph, UMAP, and Leiden Clustering

A k-nearest neighbor graph connects spots with similar expression profiles in PCA space. Leiden community detection then identifies groups of tightly connected spots. Resolution 0.5 is a conservative starting point that produces biologically interpretable cluster numbers without over-fragmenting tissue regions. UMAP provides a 2D visualization of the same structure.

In [ ]:
# Build nearest neighbor graph using top 30 PCA components
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)

# Run UMAP for 2D visualization
sc.tl.umap(adata)

# Run Leiden clustering
sc.tl.leiden(
    adata,
    resolution=0.5,
    flavor='igraph',
    n_iterations=2,
    directed=False
)

# Visualize clusters in UMAP space
sc.pl.umap(
    adata,
    color='leiden',
    save='_umap_clusters.png'
)

print(f'Leiden clustering produced {adata.obs["leiden"].nunique()} clusters')

## 6. Spatial Visualization of Clusters

Overlaying clusters on the H&E tissue image is the critical validation step. Spatially coherent clusters that align with visible anatomical boundaries support the interpretation that gene expression differences reflect tissue organization. Fragmented or anatomically incoherent clusters would suggest technical noise is dominating the signal.

In [ ]:
# Remove any stale color palette to avoid mismatch errors
adata.uns.pop('leiden_colors', None)

# Overlay clusters on H&E tissue image
sq.pl.spatial_scatter(
    adata,
    color='leiden',
    title='Leiden clusters on mouse brain tissue',
    save='spatial_clusters.png'
)

print('Spatial cluster visualization saved')

## 7. Spatially Variable Gene Detection

Moran's I measures spatial autocorrelation for each gene. A score near 1 means the gene is highly spatially structured (high expression concentrated in one region), while a score near 0 means expression is randomly distributed across the tissue. Genes with high Moran's I scores are strong candidates for region-specific markers and provide independent validation of the spatial structure identified by clustering.

In [ ]:
# Build spatial neighbor graph based on physical coordinates
sq.gr.spatial_neighbors(adata)

# Compute Moran's I for all genes
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    n_perms=100,
    n_jobs=1
)

# Display top 10 spatially variable genes
top_svgs = adata.uns['moranI'].head(10)
print('Top spatially variable genes:')
print(top_svgs)

## 8. Spatial Expression Maps for Top SVGs

Visualizing where the top spatially variable genes are expressed on the tissue confirms whether their patterns align with known anatomy. Mbp and Plp1 mark oligodendrocytes concentrated in white matter tracts. Slc17a7 and Nrgn mark excitatory neurons enriched in cortex and hippocampus. Convergence between clustering results and known marker biology strengthens confidence that clusters reflect anatomy rather than technical artifacts.

In [ ]:
# Get top 4 spatially variable genes
top_genes = adata.uns['moranI'].head(4).index.tolist()
print('Visualizing:', top_genes)

# Clear any stale color data
for gene in top_genes:
    adata.uns.pop(f'{gene}_colors', None)

# Visualize expression on tissue
sq.pl.spatial_scatter(
    adata,
    color=top_genes,
    title=top_genes,
    save='spatial_gene_map.png'
)

print('Spatial gene map saved')

## Summary and Interpretation

Leiden clustering on raw Visium spot expression profiles produced 11 spatially coherent clusters aligning with major anatomical structures including cortex, hippocampus, and white matter tracts. This confirms that unsupervised clustering can recover tissue organization without cell type deconvolution.

The top spatially variable genes by Moran's I include Nrgn (hippocampal neuron marker), Slc17a7 (excitatory neuron marker), Mbp and Plp1 (oligodendrocyte markers), and Cck (cortical interneuron marker). Convergence of these known region-specific markers with the cluster boundaries provides independent biological validation of the spatial structure.

**Critical limitation:** Each Visium spot integrates signal from multiple cell types. Clusters represent dominant expression mixtures, not pure populations. Assigning cell type identities requires deconvolution using a matched single cell RNA-seq reference.

**Next step in real research:** Run Cell2location deconvolution with the Allen Brain Atlas single cell reference, then perform neighborhood enrichment analysis with Squidpy to identify which cell types are spatially co-enriched. This is where biologically actionable hypotheses about cell-cell communication and niche organization emerge.